In [1]:
import matplotlib.pyplot as plt
plt.rcParams["font.size"] = 16

# 1.A : Model

Canva

# 1.B : Jumps (Ryu (data + fitted) + 3 trajectories (Ryu + mu2 + theta2))

In [13]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import gamma
plt.rcParams["font.size"] = 16


# ─────────────────────────────────────────────
# Conversion nm -> bp
# ─────────────────────────────────────────────
conv = (0.246 + 0.255 + 0.257) / 3  # nm per bp


def proba_gamma(mu: float, theta: float, L: float) -> float:
    alpha_gamma = mu**2 / theta**2
    beta_gamma = theta**2 / mu
    return gamma.pdf(L, a=alpha_gamma, scale=beta_gamma)


# ─────────────────────────────────────────────
# Données expérimentales (converties en bp)
# ─────────────────────────────────────────────
x_nm = np.arange(0, 150, 10)
x = x_nm / conv 
y = np.array(
    [10, 140, 240, 200, 130, 125, 90, 60, 30, 25, 20, 10, 15, 7, 5], 
    dtype=float
)
y /= np.sum(y)

# ─────────────────────────────────────────────
# Axe fin (bp)
# ─────────────────────────────────────────────
x_fine_nm = np.arange(0, 150, 1)
x_fine = x_fine_nm / conv


# ─────────────────────────────────────────────
# Fit
# ─────────────────────────────────────────────
popt, pcov = curve_fit(
    lambda x, mu, theta: proba_gamma(mu, theta, x) * (10 / conv),
    x,
    y,
    p0=[80, 40],  # en bp maintenant
    bounds=(0, np.inf),
    maxfev=10000
)

mu, theta = popt
print(f"mu={mu} and theta ={theta}")


# ─────────────────────────────────────────────
# PDF continue
# ─────────────────────────────────────────────
p = proba_gamma(mu, theta, x_fine)

bin_width = (10 / conv)  # bin en bp
p_plot = p * bin_width


# ─────────────────────────────────────────────
# Plot
# ─────────────────────────────────────────────
plt.figure(figsize=(8,6), dpi=1200)

plt.bar(
    x, y,
    width=(8 / conv),
    edgecolor="black",
    color="red",
    alpha=0.5,
    label="Data from Ryu et al. 2022"
)

plt.plot(
    x_fine,
    p_plot,
    lw=2,
    color="red",
    label=f"Gamma fit : (μ,θ) = ({150:.0f},{theta:.0f}) bp"
)

plt.xlabel("Step size (bp)")
plt.ylabel("Probability")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

mu=153.04832815111186 and theta =100.02302575647388


# 1.C : Properties of the Gamma distrib

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from nucleo.simulation.probabilities import proba_gamma

plt.rcParams["font.size"] = 16

x_fine = np.arange(0, 500, 1)
params = [
    (100, 20),
    (200, 20),
    (200, 200),
]

cmap = plt.cm.Reds
colors = [cmap(0.4), cmap(0.65), cmap(0.9)]

plt.figure(figsize=(8,6), dpi=1200)

for (mu, theta), color in zip(params, colors):
    p = proba_gamma(mu=mu, theta=theta, L=x_fine)
    
    plt.plot(
        x_fine,
        p,
        lw=2.5,
        color=color,
        label=f"(μ,θ) = ({mu},{theta})"
    )

plt.xlabel(r"Step size $(\sigma)$")
plt.ylabel("Probability density")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", ncols=1)
plt.tight_layout()
plt.show()

[0.00000000e+00 1.11486152e-39 1.45669021e-32 1.90978201e-28
 1.48231279e-25 2.44459291e-23 1.51350142e-21 4.76575995e-20
 9.14882395e-19 1.20351689e-17 1.17505540e-16 9.01382589e-16
 5.66579695e-15 3.01285295e-14 1.38943014e-13 5.66735551e-13
 2.07728556e-12 6.93133855e-12 2.12818560e-11 6.06711435e-11
 1.61823665e-10 4.06454176e-10 9.66761253e-10 2.18811095e-09
 4.73257447e-09 9.81780140e-09 1.95993093e-08 3.77604476e-08
 7.03924248e-08 1.27265739e-07 2.23612484e-07 3.82554191e-07
 6.38319587e-07 1.04039829e-06 1.65876728e-06 2.59030494e-06
 3.96647049e-06 5.96227450e-06 8.80649894e-06 1.27930447e-05
 1.82931931e-05 2.57684721e-05 3.57837175e-05 4.90198299e-05
 6.62856442e-05 8.85282656e-05 1.16841188e-04 1.52469500e-04
 1.96811495e-04 2.51416088e-04 3.17975477e-04 3.98312677e-04
 4.94363649e-04 6.08153950e-04 7.41770020e-04 8.97325415e-04
 1.07692249e-03 1.28261028e-03 1.51633935e-03 1.77991482e-03
 2.07494853e-03 2.40281170e-03 2.76458931e-03 3.16103747e-03
 3.59254496e-03 4.059100

# 1.D : Chromatin lanscapes (homogeneous + periodic + random)

In [5]:
from nucleo.simulation.chromatin import alpha_random, alpha_periodic, alpha_homogeneous
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams["font.size"] = 16

# Values in nm
s = 150
l = 10
alphao = 0
alphaf = 1
Lmin = 0
Lmax = 500
bps = 1

# Landscapes
obs_1 = alpha_homogeneous(s=s, l=l, alphao=alphao, alphaf=alphaf, Lmin=0, Lmax=Lmax, bps=bps)
obs_2 = alpha_periodic(s=s, l=l, alphao=alphao, alphaf=alphaf, Lmin=0, Lmax=Lmax, bps=bps)
obs_3 = alpha_random(s=s, l=l, alphao=alphao, alphaf=alphaf, Lmin=0, Lmax=Lmax, bps=bps)
x = np.arange(0, len(obs_1), 1)

# -------------------------------
# Homogeneous
# -------------------------------
plt.figure(figsize=(8,5), dpi=1200)

plt.plot(x, obs_1, c="b", lw=3)
plt.title("Homogeneous")
plt.xlabel("x")
plt.ylabel(r"Accessibility $\alpha$")
plt.ylim([-0.1, 1.1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# -------------------------------
# Periodic
# -------------------------------
plt.figure(figsize=(8,5), dpi=1200)

plt.step(x, obs_2, c="b", lw=3)
plt.title("Periodic")
plt.xlabel("x")
# plt.ylabel(r"Accessibility $\alpha$")
plt.ylim([-0.1, 1.1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# -------------------------------
# Random
# -------------------------------
plt.figure(figsize=(8,5), dpi=1200)

plt.step(x, obs_3, c="b", lw=3)
plt.title("Random")
plt.xlabel("x")
# plt.ylabel(r"Accessibility $\alpha$")
plt.ylim([-0.1, 1.1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# 1.E : Linker and RoadBlocks distribution

In [42]:
# Librairies
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
from polars import selectors as cs
from pathlib import Path
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from nucleo.metrics.utils import listoflist_into_matrix
plt.rcParams["font.size"] = 16

# Data
root = Path("/home/nicolas/Documents/Workspace/nucleo/outputs/2026-03-12__PC/nucleo__fig1_0")
paths = [str(p) for p in root.rglob("*.parquet")]
df_sorted = (
    pl.scan_parquet(paths)
    .select(
        cs.string() | cs.boolean() |  cs.integer() |  cs.float() | 
        pl.col("s_points") | pl.col("s_distrib") | pl.col("l_points") | pl.col("l_distrib")
      )
    .collect()
    .sort(by=["landscape", "bpmin", "l"],
          descending=[False, False, False]
        )
      .filter(
          pl.col("landscape") == "random"
      )
)

# Obstacles
obs_points_data = df_sorted["s_points"].to_list()
obs_points_data = listoflist_into_matrix(obs_points_data)
obs_points = np.nanmean(obs_points_data,axis=0)

obs_distrib_data = df_sorted["s_distrib"].to_list()
obs_distrib_data = listoflist_into_matrix(obs_distrib_data)
obs_distrib = np.nanmean(obs_distrib_data,axis=0)

# To roadblocks
mask_s = (obs_distrib != 0)
obs_points = obs_points[mask_s]
obs_distrib = obs_distrib[mask_s]
roadblocks_points = obs_points//s
roadblocks_distrib = obs_distrib

# Linkers
link_points_data = df_sorted["l_points"].to_list()
link_points_data = listoflist_into_matrix(link_points_data)
link_points = np.nanmean(link_points_data,axis=0)
link_points = np.concatenate((np.array([0.0]), link_points))

link_distrib_data = df_sorted["l_distrib"].to_list()
link_distrib_data = listoflist_into_matrix(link_distrib_data)
link_distrib = np.nanmean(link_distrib_data,axis=0)


# Theory
from scipy.special import factorial

def th_linkers(l, lmoy):
    A = lmoy * (1 - np.exp(-1 / lmoy))
    P = A * (np.exp(1 / lmoy) - 1) * np.exp(-l / lmoy)
    P = P / np.sum(P)
    P0 = np.array([1 - A])
    PF = np.concatenate((P0, P))
    return PF / np.sum(PF)


def th_roadblocks(m, lmoy):
    m = np.atleast_1d(m).astype(int)
    A = np.exp(1 / lmoy) - 1
    P = np.array([1 / (lmoy**mi * A * factorial(mi)) for mi in m], dtype=float)
    return P / np.sum(P)


# Plot
fig, axes = plt.subplots(ncols=1 , nrows=2, figsize=(8,6), dpi=1200)

s = 150
lmoy = 10

theory_distrib = th_linkers(link_points[1:], lmoy)

axes[0].plot(link_points[:-1], link_distrib, label='Simulation',
             color='lightblue', alpha=1, marker='o', lw=2)
axes[0].plot(link_points, theory_distrib, label="Theory", 
             color='black', ls="--", lw=2)

axes[0].set_xlabel('Size of linkers')
axes[0].set_ylabel('Distribution')
axes[0].set_xlim([-1, 45])
# axes[0].set_ylim([-0.10, 0.25])
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

print(link_distrib, link_points)



theory_distrib = th_roadblocks(roadblocks_points, lmoy)

axes[1].plot(roadblocks_points, roadblocks_distrib, label='Simulation',
      color='darkslategray', alpha=1, marker='o', lw=2)
axes[1].plot(roadblocks_points, theory_distrib, label="Theory",
             color='black', ls="--", lw=2)

axes[1].set_xticks(np.arange(1, 6, 1, dtype=int))
axes[1].set_ylim([-0.10, 1.10])
axes[1].set_xlabel("Consecutive roadblocks")
axes[1].set_ylabel('Distribution')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

[4.6827082e-02 8.8722631e-02 8.0558732e-02 7.2359070e-02 6.6503286e-02
 5.9953202e-02 5.4158092e-02 4.9679540e-02 4.5018729e-02 4.0255636e-02
 3.7218623e-02 3.3257607e-02 3.0093385e-02 2.7366307e-02 2.5048906e-02
 2.2587080e-02 2.0807266e-02 1.8402750e-02 1.6944945e-02 1.5452849e-02
 1.4186904e-02 1.2521457e-02 1.1092353e-02 1.0389108e-02 9.1869906e-03
 8.6181331e-03 7.9241702e-03 7.1097086e-03 6.3196351e-03 5.6485543e-03
 5.3670341e-03 4.6669617e-03 4.2549502e-03 3.7462083e-03 3.5387971e-03
 3.1645359e-03 3.0018527e-03 2.5987965e-03 2.2857725e-03 2.0458994e-03
 2.0297833e-03 1.8318720e-03 1.6862563e-03 1.4908767e-03 1.3544190e-03
 1.2522951e-03 1.1551331e-03 8.9643622e-04 1.0160550e-03 9.3739585e-04
 7.9486368e-04 7.5679837e-04 5.9929333e-04 5.9575931e-04 5.9557142e-04
 5.6503795e-04 5.8575952e-04 5.6150235e-04 4.6953722e-04 4.5522751e-04
 4.4825338e-04 3.7207222e-04 3.8210652e-04 4.1010146e-04 4.1224205e-04
 4.5554273e-04 3.6576710e-04 4.2943959e-04 3.6101096e-04 4.1795959e-04
 3.918

# .